In [45]:
!pip install --quiet "openai>=1.0.0"

from openai import OpenAI
import json

client = OpenAI(api_key="YOUR_GROQ_API_KEY", base_url="https://api.groq.com/openai/v1")


In [46]:
# Task 1: Conversation History with Summarization

In [47]:

conversation_history = []

def summarize_conversation(history, max_tokens=150):
    """
    Summarize the entire conversation so far.
    """
    conversation_text = "\n".join([f"{msg['role']}: {msg['content']}" for msg in history])

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "Summarize the conversation concisely."},
            {"role": "user", "content": conversation_text}
        ],
        max_tokens=max_tokens
    )

    return response.choices[0].message.content



def truncate_history(history, by_turns=None, by_chars=None):
    """
    Truncate conversation history either by turns or by character count.
    """
    if by_turns:
        history = history[-by_turns:]
    if by_chars:
        truncated = []
        total_chars = 0
        for msg in reversed(history):
            total_chars += len(msg['content'])
            if total_chars > by_chars:
                break
            truncated.insert(0, msg)
        history = truncated
    return history


def add_message(role, content, history, k=3):
    """
    Add new message and perform periodic summarization after k messages.
    """
    history.append({"role": role, "content": content})
    if len(history) % k == 0:
        summary = summarize_conversation(history)
        history = [{"role": "system", "content": f"Summary so far: {summary}"}]
    return history


In [48]:
conversation_history = add_message("user", "Hi there!", conversation_history)
conversation_history = add_message("assistant", "Hello! How are you?", conversation_history)
conversation_history = add_message("user", "I want to know about Artificial Intelligence.", conversation_history)

print("Conversation (before truncation):")
print(conversation_history)


print("\nTruncated (last 2 turns):")
print(truncate_history(conversation_history, by_turns=2))


conversation_history = add_message("assistant", "Sure, AI is a vast field involving machine learning.", conversation_history, k=3)

print("\nConversation after 3rd run summarization:")
print(conversation_history)


Conversation (before truncation):
[{'role': 'system', 'content': "Summary so far: There's no conversation history to summarize. Let's start the conversation about Artificial Intelligence.\n\nArtificial Intelligence (AI) refers to the development of computer systems that can perform tasks that would typically require human intelligence, such as learning, problem-solving, decision-making, and perception. What aspect of AI would you like to know about?"}]

Truncated (last 2 turns):
[{'role': 'system', 'content': "Summary so far: There's no conversation history to summarize. Let's start the conversation about Artificial Intelligence.\n\nArtificial Intelligence (AI) refers to the development of computer systems that can perform tasks that would typically require human intelligence, such as learning, problem-solving, decision-making, and perception. What aspect of AI would you like to know about?"}]

Conversation after 3rd run summarization:
[{'role': 'system', 'content': "Summary so far: Th

In [49]:
#Task 2: JSON Schema Classification & Extraction

In [50]:
def extract_user_info(chat_text):
    """
    Extract structured user details using Groq API with function calling.
    """
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": chat_text}],
        tools=[
            {
                "type": "function",
                "function": {
                    "name": "extract_user_info",
                    "description": "Extract user details from text",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string"},
                            "email": {"type": "string"},
                            "phone": {"type": "string"},
                            "location": {"type": "string"},
                            "age": {"type": "string"} # Changed to string
                        },
                        "required": ["name", "email", "phone", "location", "age"]
                    }
                }
            }
        ],
        tool_choice={"type": "function", "function": {"name": "extract_user_info"}}
    )

    args = response.choices[0].message.tool_calls[0].function.arguments
    return json.loads(args)

In [51]:
sample_chats = [
    "Hi, my name is mohammad , I live in delhi. You can contact me at mohammad@example.com or 1234567890. I am 25 years old.",
    "Hello, alexa here from London, age 30. Email: alexa@example.com, phone: 9876543210.",
    "Hey, siri from Sydney. Contact: siri@sydney.com, 5555555555, age 28."
]

for chat in sample_chats:
    info = extract_user_info(chat)
    print("Input:", chat)
    print("Extracted:", info, "\n")


Input: Hi, my name is mohammad , I live in delhi. You can contact me at mohammad@example.com or 1234567890. I am 25 years old.
Extracted: {'age': '25 years old', 'email': 'mohammad@example.com', 'location': 'delhi', 'name': 'mohammad', 'phone': '1234567890'} 

Input: Hello, alexa here from London, age 30. Email: alexa@example.com, phone: 9876543210.
Extracted: {'age': '30', 'email': 'alex@example.com', 'location': 'London', 'name': 'alexa', 'phone': '9876543210'} 

Input: Hey, siri from Sydney. Contact: siri@sydney.com, 5555555555, age 28.
Extracted: {'age': '28', 'email': 'siri@sydney.com', 'location': 'Sydney', 'name': 'siri', 'phone': '5555555555'} 

